DAILY NEWS INFRA - LINK: 
https://ppi.gov.br/projetos/

In [0]:
%pip install --quiet beautifulsoup4 curl_cffi httpx h2 lxml playwright

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%sh playwright install chromium --with-deps

Installing dependencies...
Switching to root user to install dependencies...
sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: a password is required
Failed to install browsers
Error: Installation process exited with code: 1


In [0]:
dbutils.library.restartPython()

In [0]:
# ==============================================================================
# Etapa A — Configuração
# ==============================================================================
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests
from playwright.sync_api import sync_playwright

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

SITE_URL = "https://ppi.gov.br/noticias/"

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/sites/{HOJE}"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

DEBUG = True

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4 Safari/605.1.15",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124", "safari17_0", "edge101"]

HTTP_TIMEOUT = 30


# ==============================================================================
# Etapa B — Helpers
# ==============================================================================
def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def domain_from_url(url: str) -> str:
    try:
        netloc = urllib.parse.urlparse(url).netloc.lower()
        return netloc[4:] if netloc.startswith("www.") else netloc
    except Exception:
        return "desconhecido"


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    ua = random.choice(USER_AGENTS)
    headers = {
        "User-Agent": ua,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,"
                  "image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1",
    }
    if referer:
        headers["Referer"] = referer
    return headers


# ==============================================================================
# Etapa 1 — Baixar o HTML com técnicas anti-bot
# ==============================================================================
# Adicionar esse import junto com os outros, no topo do script:
import concurrent.futures


def _baixar_html_playwright_worker(url: str, timeout_ms: int) -> Optional[str]:
    """
    Função que roda de fato o Playwright (Sync API). Precisa ser chamada
    de dentro de uma thread separada (ver baixar_html_playwright abaixo),
    porque o notebook do Databricks já roda um loop asyncio por padrão,
    e o Playwright Sync API não pode rodar dentro de um loop já existente.
    """
    from playwright.sync_api import sync_playwright
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            context = browser.new_context(
                user_agent=random.choice(USER_AGENTS),
                locale="pt-BR",
                viewport={"width": 1366, "height": 768},
            )
            page = context.new_page()
            page.goto(url, timeout=timeout_ms, wait_until="networkidle")
            page.wait_for_timeout(3000)
            html = page.content()
            browser.close()
            return html
    except Exception as e:
        print(f"  [playwright] erro: {e}")
        return None


def baixar_html_playwright(url: str, timeout_ms: int = 45000) -> Optional[str]:
    """
    Plano C: baixa a página usando um navegador Chrome real (headless), via
    Playwright. Roda numa thread separada para evitar conflito com o loop
    asyncio que o notebook do Databricks já mantém ativo.
    """
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_baixar_html_playwright_worker, url, timeout_ms)
        return future.result()


def parece_conteudo_valido(html: str, min_links: int = 3) -> bool:
    """
    Verificação extra: uma página de listagem de notícias de verdade deve
    ter vários links <a href>. Páginas de desafio anti-bot (tipo a do
    ppi.gov.br) costumam vir "vazias" nesse sentido, mesmo tendo bastante
    HTML (por causa do script de desafio). Isso evita aceitar por engano
    uma página de bloqueio só porque ela passou no teste de tamanho.
    """
    if not html:
        return False
    return html.count("<a ") >= min_links


def baixar_html(url: str, tentativas: int = 3) -> Optional[str]:
    for i in range(1, tentativas + 1):
        time.sleep(random.uniform(0.8, 2.2))

        headers = headers_aleatorios(referer="https://www.google.com/")
        impersonate = random.choice(IMPERSONATE_PROFILES)

        # --- Plano A: curl_cffi ---
        try:
            resp = cffi_requests.get(
                url, headers=headers, impersonate=impersonate,
                timeout=HTTP_TIMEOUT, allow_redirects=True,
            )
            if (200 <= resp.status_code < 300 and resp.text
                    and len(resp.text) > 500 and parece_conteudo_valido(resp.text)):
                return resp.text
            print(f"  [curl_cffi tent {i}] status={resp.status_code} "
                  f"len={len(resp.text) if resp.text else 0} "
                  f"(conteudo_valido={parece_conteudo_valido(resp.text) if resp.text else False})")
            if DEBUG:
                print(f"    [debug] primeiros 300 chars: {resp.text[:300]!r}")
        except Exception as e:
            print(f"  [curl_cffi tent {i}] erro: {e}")

        # --- Plano B: httpx ---
        try:
            with httpx.Client(
                headers=headers, follow_redirects=True,
                timeout=HTTP_TIMEOUT, http2=True,
            ) as client:
                resp = client.get(url)
                if (200 <= resp.status_code < 300 and resp.text
                        and len(resp.text) > 500 and parece_conteudo_valido(resp.text)):
                    return resp.text
                print(f"  [httpx tent {i}] status={resp.status_code} "
                      f"len={len(resp.text) if resp.text else 0} "
                      f"(conteudo_valido={parece_conteudo_valido(resp.text) if resp.text else False})")
                if DEBUG:
                    print(f"    [debug] primeiros 300 chars: {resp.text[:300]!r}")
        except Exception as e:
            print(f"  [httpx tent {i}] erro: {e}")

    # --- Plano C: navegador real (Playwright) ---
    print("  -> curl_cffi/httpx falharam (ou devolveram página de bloqueio) "
          "em todas as tentativas; tentando com navegador headless (Playwright)...")
    html = baixar_html_playwright(url)
    if html and len(html) > 500 and parece_conteudo_valido(html):
        print(f"  -> Playwright conseguiu! HTML com {len(html)} chars.")
        return html
    print("  -> Playwright também falhou (ou devolveu página de bloqueio).")

    return None

# ==============================================================================
# Etapa 2 — Extrair todos os links da página
# ==============================================================================
def extrair_links(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    links = []
    vistos = set()

    for tag_a in soup.find_all("a", href=True):
        href = tag_a["href"].strip()

        if (not href or href.startswith("#") or href.startswith("javascript:")
                or href.startswith("mailto:") or href.startswith("tel:")):
            continue

        url_absoluta = urllib.parse.urljoin(url_base, href)

        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)

        texto_ancora = tag_a.get_text(" ", strip=True)
        links.append({"texto_ancora": texto_ancora, "url": url_absoluta})

    return links


# ==============================================================================
# Etapa 3 — Limpar o HTML e extrair só o texto útil
# ==============================================================================
TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form",
             "nav", "footer", "header", "aside", "button"]


def extrair_texto(html: str) -> str:
    if not html:
        return ""

    soup = BeautifulSoup(html, "lxml")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    articles = soup.find_all("article")

    if len(articles) == 1:
        texto_article = articles[0].get_text("\n", strip=True)
        if len(texto_article) > 500:
            texto = texto_article
        else:
            texto = soup.get_text("\n", strip=True)
    else:
        texto = soup.get_text("\n", strip=True)

    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


def extrair_titulo(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
        if soup.title and soup.title.string:
            return soup.title.string.strip()
    except Exception:
        pass
    return ""


# ==============================================================================
# Etapa 4 — Salvar no Volume
# ==============================================================================
def salvar_artefatos(pasta, source, titulo, texto, links, metadados):
    slug_source = slugify(source, max_len=40) or "fonte"
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_links = os.path.join(pasta, f"{nome_base}_links.json")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")

    with open(caminho_links, "w", encoding="utf-8") as f:
        json.dump(links, f, ensure_ascii=False, indent=2)

    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_links, caminho_json


# ==============================================================================
# Etapa 5 — Pipeline principal
# ==============================================================================
def processar_site(url: str, pasta_destino: str) -> Optional[dict]:
    print(f"\n=== Site: {url!r} ===")

    html = baixar_html(url)
    if not html:
        print("  -> download do HTML falhou (mesmo com Playwright); abortando.")
        return None
    print(f"  HTML baixado ({len(html)} chars).")

    links = extrair_links(html, url_base=url)
    print(f"  {len(links)} links únicos encontrados.")

    texto = extrair_texto(html)
    print(f"  Texto extraído ({len(texto)} chars).")

    source = domain_from_url(url)
    titulo = extrair_titulo(html) or source
    metadados = {
        "source_id": "site_page",
        "title": titulo,
        "description": f"Página coletada diretamente do site: {url}",
        "url": url,
        "date": HOJE,
        "qtd_links": len(links),
        "qtd_chars_texto": len(texto),
    }

    caminho_txt, caminho_links, caminho_json = salvar_artefatos(
        pasta=pasta_destino, source=source, titulo=titulo,
        texto=texto, links=links, metadados=metadados,
    )
    print(f"  -> salvo em {caminho_txt}")
    print(f"  -> links em {caminho_links}")

    return {
        "url": url, "titulo": titulo, "source": source,
        "qtd_links": len(links), "qtd_chars_texto": len(texto),
        "caminho_txt": caminho_txt, "caminho_links": caminho_links,
        "caminho_json": caminho_json,
    }


# ==============================================================================
# Execução
# ==============================================================================
resultado = processar_site(SITE_URL, PASTA_DESTINO)

if resultado:
    print(f"\n\n=== Fim. Site processado com sucesso ===")
    print(json.dumps(resultado, ensure_ascii=False, indent=2))
else:
    print("\n\n=== Fim. Falha ao processar o site. ===")

[setup] Salvando artefatos em: /Volumes/desafio_kinea/research/research_volume/infraestrutura/sites/2026-07-24

=== Site: 'https://ppi.gov.br/noticias/' ===
  [curl_cffi tent 1] status=200 len=6132 (conteudo_valido=False)
    [debug] primeiros 300 chars: '<!DOCTYPE html>\r\n<html><head>\r\n<meta http-equiv="Pragma" content="no-cache"/>\r\n<meta http-equiv="Expires" content="-1"/>\r\n<meta http-equiv="CacheControl" content="no-cache"/>\r\n<meta http-equiv="Content-Type" content="text/html; charset=utf-8"/>\r\n<link rel="shortcut icon" href="data:;base64,iVBORw0KG'
  [httpx tent 1] status=200 len=6825 (conteudo_valido=False)
    [debug] primeiros 300 chars: '<!DOCTYPE html>\r\n<html><head>\r\n<meta http-equiv="Pragma" content="no-cache"/>\r\n<meta http-equiv="Expires" content="-1"/>\r\n<meta http-equiv="CacheControl" content="no-cache"/>\r\n<meta http-equiv="Content-Type" content="text/html; charset=utf-8"/>\r\n<link rel="shortcut icon" href="data:;base64,iVBORw0KG'
  [curl_cffi tent 2] 